# MCP Gateway Interactive Exploration

Explore the MCP Gateway's tool catalog interactively before running the full
SDG generation pipeline. This notebook demonstrates the session handshake,
tool discovery via `tools/list`, and sample tool invocations.

**Prerequisites:**
- MCP Gateway is reachable (set `GATEWAY_URL`, `KEYCLOAK_URL`, `CLIENT_SECRET`)
- This notebook is intended to run in an RHOAI workbench pointed at the `OTel_agentic_improvements` repo

**Purpose:** Understand the available tool catalog, inspect schemas, and
test individual tool calls before committing to generation.

In [ ]:
import sys
import os
import json

# Add repo root to path for proper package imports
sys.path.insert(0, os.path.abspath('../..'))

try:
    import requests
except ImportError:
    !pip install requests
    import requests

# Configuration from environment
GATEWAY_URL = os.environ.get(
    'GATEWAY_URL',
    'https://mcp.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com/mcp'
)
KEYCLOAK_URL = os.environ.get(
    'KEYCLOAK_URL',
    'https://mcp-keycloak-otel25-gateway.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com'
)
CLIENT_ID = os.environ.get('CLIENT_ID', 'sdg-pipeline')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')

if not CLIENT_SECRET:
    raise ValueError('Set CLIENT_SECRET env var for MCP Gateway authentication')

print(f'Gateway:  {GATEWAY_URL}')
print(f'Keycloak: {KEYCLOAK_URL}')
print(f'Client:   {CLIENT_ID}')

In [ ]:
# ── Step 1: Acquire JWT + MCP Session Handshake ──
# The MCP Gateway requires:
#   1. A JWT token from Keycloak (client_credentials grant)
#   2. An MCP initialize handshake to obtain Mcp-Session-Id
#   3. The Mcp-Session-Id header on all subsequent requests

from sdg.scripts.materialize_seeds import acquire_jwt, mcp_initialize

# Acquire JWT
jwt = acquire_jwt(KEYCLOAK_URL, CLIENT_ID, CLIENT_SECRET)
print(f'JWT acquired ({len(jwt)} chars)')

# MCP initialize handshake
session_id = mcp_initialize(GATEWAY_URL, jwt)
print(f'Mcp-Session-Id: {session_id}')

# Build reusable headers for subsequent requests
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json, text/event-stream',
    'Authorization': f'Bearer {jwt}',
    'Mcp-Session-Id': session_id,
}

In [ ]:
# ── Step 2: List all tools via tools/list ──

tools_response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={'jsonrpc': '2.0', 'method': 'tools/list', 'id': 2},
    verify=False,
)

tools_data = tools_response.json()
tools = tools_data.get('result', {}).get('tools', [])

print(f'Total tools available: {len(tools)}')
print()

# Group by domain prefix
domains = {}
for tool in tools:
    name = tool['name']
    prefix = name.split('_')[0] if '_' in name else name.split('-')[0]
    domains.setdefault(prefix, []).append(name)

print('Tools by domain prefix:')
for domain, tool_names in sorted(domains.items()):
    print(f'  {domain}: {len(tool_names)} tools')
    for t in sorted(tool_names)[:5]:
        print(f'    - {t}')
    if len(tool_names) > 5:
        print(f'    ... and {len(tool_names) - 5} more')

In [ ]:
# ── Step 3: Call a sample tool (tmf642_list_alarms) ──

call_response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        'jsonrpc': '2.0',
        'method': 'tools/call',
        'params': {
            'name': 'tmf642_list_alarms',
            'arguments': {},
        },
        'id': 3,
    },
    verify=False,
)

call_data = call_response.json()
print('tmf642_list_alarms response:')
print(json.dumps(call_data, indent=2)[:2000])

In [ ]:
# ── Step 4: Inspect tool schemas ──
# Show input schemas for tools used in the dry-run (single_tool tier)

dry_run_tools = [
    'tmf642_list_alarms',
    'tmf628_list_performance_jobs',
    'tmf621_create_trouble_ticket',
    'get_nwdaf_analytics',
    'camara_verify_location',
]

tool_map = {t['name']: t for t in tools}

for name in dry_run_tools:
    tool = tool_map.get(name)
    if tool:
        print(f'\n=== {name} ===')
        print(f'Description: {tool.get("description", "N/A")}')
        schema = tool.get('inputSchema', {})
        print(f'Input schema: {json.dumps(schema, indent=2)[:500]}')
    else:
        print(f'\n=== {name} === NOT FOUND in gateway')

## Summary

This notebook demonstrated:
1. JWT acquisition from Keycloak using `client_credentials` grant
2. MCP session handshake (`initialize` + `Mcp-Session-Id`)
3. Tool discovery via `tools/list`
4. Individual tool invocation via `tools/call`
5. Schema inspection for dry-run tools

Use this as a reference when debugging tool availability or schema mismatches
during the generation pipeline.